# Machine Learning Models: Cross-Sectional Price Prediction

## Learning Goals

After completing this notebook, you will be able to:

- Implement multiple regression algorithms (Linear, Ridge, LASSO, Random Forest)
- Understand the trade-offs between model complexity and interpretability
- Train and evaluate models using train-test splits
- Interpret model coefficients to understand feature importance
- Compare model performance using appropriate metrics (R-squared, RMSE)

## Keywords

machine learning, regression, model selection, feature importance, regularization

## Prerequisite Knowledge

02_exploratory_analysis.ipynb (understanding of data patterns)

## Target User

Aspiring machine learning practitioners learning to build and compare regression models

## Table of Contents

1. Part 1: Linear Regression Baseline
2. Part 2: Regularized Models (Ridge and LASSO)
3. Part 3: Ensemble Methods (Random Forest)
4. Part 4: Model Comparison

## Part 1: Linear Regression Baseline

### Why Start with Linear Regression?

Linear regression is the foundation of predictive modeling:
- **Interpretable**: Each feature gets a coefficient showing its effect on price
- **Fast**: Computes quickly on any size dataset
- **Baseline**: Provides a reference point - fancier models should beat it
- **Assumptions transparent**: You know when it might fail

Even though housing prices have complex patterns (non-linear), a linear model shows what would work if relationships were purely linear.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler

# California housing serves as a stand-in for Ames when running notebook standalone
data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target * 100000

print(f'Dataset shape: {X.shape}')
print(f'Features: {list(X.columns)}')
print(f'Price range: ${y.min():,.0f} to ${y.max():,.0f}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred_test = linear_model.predict(X_test)
linear_r2 = r2_score(y_test, y_pred_test)
linear_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

print('Linear Regression Results:')
print(f'  Test R-squared: {linear_r2:.3f}')
print(f'  Test RMSE: ${linear_rmse:,.0f}')

coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': linear_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)
print('\nTop features by coefficient magnitude:')
print(coef_df.head(5).to_string(index=False))

### Concept Check 1.1

A linear model achieves R-squared = 0.60 on both training and test sets. What does this indicate?

A) Perfect model - no overfitting
B) Model captures 60% of variance and generalizes well
C) Model is underfitting and needs more features
D) Model should not be deployed

<details>
<summary>Answer</summary>
B) Model captures 60% of variance and generalizes well. Equal train/test R-squared means no overfitting.
</details>

## Part 2: Regularized Models (Ridge and LASSO)

### The Problem with Linear Regression

Linear regression can overfit when:
- You have many features (some might fit noise, not signal)
- Features are highly correlated (multicollinearity)
- Dataset is small relative to number of features

**Regularization** adds a penalty for large coefficients:
- **Ridge (L2)**: Shrinks all coefficients toward zero, keeps all features
- **LASSO (L1)**: Can shrink coefficients exactly to zero (feature selection)

In [ ]:
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train)
ridge_pred = ridge_model.predict(X_test_scaled)
ridge_r2 = r2_score(y_test, ridge_pred)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))

print('Ridge Regression (alpha=1.0):')
print(f'  Test R-squared: {ridge_r2:.3f}')
print(f'  Test RMSE: ${ridge_rmse:,.0f}')

In [ ]:
lasso_model = Lasso(alpha=100)
lasso_model.fit(X_train_scaled, y_train)
lasso_pred = lasso_model.predict(X_test_scaled)
lasso_r2 = r2_score(y_test, lasso_pred)
lasso_rmse = np.sqrt(mean_squared_error(y_test, lasso_pred))

print('LASSO Regression (alpha=100):')
print(f'  Test R-squared: {lasso_r2:.3f}')
print(f'  Test RMSE: ${lasso_rmse:,.0f}')
print(f'  Non-zero features: {(lasso_model.coef_ != 0).sum()} of {len(lasso_model.coef_)}')

### Concept Check 2.1

When should you prefer LASSO over Ridge?

A) Always - simpler is better
B) When you suspect many features are irrelevant
C) When you have a small dataset
D) Never - Ridge is always more accurate

<details>
<summary>Answer</summary>
B) When you suspect many features are irrelevant. LASSO's ability to zero out coefficients means automatic feature selection.
</details>

## Part 3: Ensemble Methods (Random Forest)

### Beyond Linear Models

Linear models assume price changes proportionally with each feature. Real relationships are often non-linear:
- Diminishing returns (extra bedroom matters less in a mansion)
- Interactions (waterfront + large lot together > sum of individual effects)
- Thresholds (below minimum size, homes don't sell at all)

**Random Forest** builds many decision trees on random subsets of data and averages their predictions - capturing non-linearity automatically.

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_r2 = r2_score(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))

print('Random Forest Results:')
print(f'  Test R-squared: {rf_r2:.3f}')
print(f'  Test RMSE: ${rf_rmse:,.0f}')

importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)
print('\nFeature Importance:')
print(importance_df.to_string(index=False))

### Concept Check 3.1

Random Forest achieves higher training R-squared than test R-squared (e.g., 0.95 vs 0.80). Is this a problem?

A) Yes - the model is overfitting badly
B) No - trees naturally fit training data tightly; ensemble averaging prevents test overfitting
C) Depends on whether test performance is good enough for your use case
D) Only if test R-squared is below 0.5

<details>
<summary>Answer</summary>
B and C. Individual trees achieve near-100% on training data by memorizing. Averaging 100 trees prevents this from hurting test performance. A gap is normal for tree ensembles.
</details>

## Part 4: Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge', 'LASSO', 'Random Forest'],
    'Test R-squared': [linear_r2, ridge_r2, lasso_r2, rf_r2],
    'Test RMSE': [linear_rmse, ridge_rmse, lasso_rmse, rf_rmse]
})

results_display = results.copy()
results_display['Test RMSE'] = results_display['Test RMSE'].apply(lambda x: f'${x:,.0f}')
results_display['Test R-squared'] = results_display['Test R-squared'].round(3)

print('Model Comparison:')
print(results_display.to_string(index=False))

best_idx = results['Test R-squared'].idxmax()
print(f'\nBest performer: {results.loc[best_idx, "Model"]}')

### When to Use Each Model

| Use Case | Best Model | Why |
|----------|-----------|-----|
| Explain to stakeholders | Linear or LASSO | Coefficients are directly interpretable |
| Highest accuracy | Random Forest | Handles non-linearity |
| Many irrelevant features | LASSO | Automatic feature selection |
| Fast real-time predictions | Linear or Ridge | Simple math operations |
| Understanding feature importance | Random Forest | Provides importance scores |

### Concept Check 4.1

Random Forest achieves R-squared = 0.70 vs. Linear Regression's 0.58. What primarily explains this improvement?

A) Random Forest uses more features
B) Random Forest captures non-linear relationships and interactions
C) Random Forest handles missing data better
D) Random Forest is a newer algorithm

<details>
<summary>Answer</summary>
B) Random Forest captures non-linear relationships and interactions. Both models use the same features, but Random Forest can model non-linear patterns and feature interactions automatically.
</details>

---

## Summary

We trained four types of models to predict housing prices:
- **Linear Regression**: Baseline, interpretable, limited by linearity assumption
- **Ridge/LASSO**: Regularized linear, prevent overfitting, interpretability preserved
- **Random Forest**: Captures non-linearity, best accuracy, less interpretable

For housing prices with complex non-linear drivers, Random Forest typically wins on accuracy. But choose based on your priorities: interpretability, speed, or accuracy.

The next notebook (04_time_series_methods.ipynb) explores whether time-based approaches can capture patterns cross-sectional models miss.

---

## Next Steps

Proceed to [04_time_series_methods.ipynb](04_time_series_methods.ipynb) to compare with time series modeling approaches.